# Creating Explanation Engine that describes "Why did you get this movies"?  

## 1. Import Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import random
import warnings

In [3]:
# Loading All Previous Components

# Enriched movies
enriched_path = '../data/processed/enriched_movies.csv'
enriched_movies = pd.read_csv(enriched_path)

# Embeddings (for content explanations)
embeddings_path = '../artifacts/content/movie_embeddings.npy'
movie_embeddings = np.load(embeddings_path)

# SVD model
model_path = '../artifacts/svd_model.pkl'
with open(model_path, 'rb') as f:
    svd_model = pickle.load(f)

# Ratings
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')

print(f"   Loaded {len(enriched_movies)} movies with explanations ready!")

   Loaded 9742 movies with explanations ready!


## 2. Generate Explanation

In [4]:
# Building an explanation generator.

def get_movie_by_title(title):
    """Find movie by partial title match."""
    matches = enriched_movies[enriched_movies['title'].str.contains(title, case=False, na=False)]
    if len(matches) > 0:
        return matches.iloc[0]
    return None

def generate_explanation(rec_row, user_id, seed_movie_title=None, alpha=0.6):
    """
    Generate a friendly, natural language explanation for a recommendation.
    """
    title = rec_row['title']
    hybrid_score = rec_row.get('hybrid_score', 0)
    collab_pred = rec_row.get('collab_pred', None)
    content_score = rec_row.get('content_score', None)
    
    explanation_parts = []
    
    # 1. Hybrid / Overall reason
    score_percent = int(hybrid_score * 100)
    explanation_parts.append(f"**Strong recommendation** with a hybrid score of {score_percent}%.")
    
    # 2. Collaborative explanation (SVD)
    if collab_pred is not None:
        if collab_pred >= 4.0:
            explanation_parts.append(f"Users with similar taste to you (user {user_id}) rated similar movies very highly — we predict you'd give this **{collab_pred}/5**.")
        elif collab_pred >= 3.5:
            explanation_parts.append(f"Based on your rating history, the model predicts you'd enjoy this with a solid **{collab_pred}/5** rating.")
        else:
            explanation_parts.append(f"Predicted rating: **{collab_pred}/5** based on collaborative patterns.")
    
    # 3. Content-based explanation (if seed movie)
    if seed_movie_title and content_score is not None and content_score > 0.6:
        seed_movie = get_movie_by_title(seed_movie_title)
        if seed_movie is not None:
            explanation_parts.append(f"It shares strong semantic themes with **{seed_movie_title}** (content similarity: {int(content_score*100)}%).")
            
            # Add genre overlap if possible
            if pd.notna(seed_movie.get('genres')) and pd.notna(rec_row.get('genres', '')):
                seed_genres = set(seed_movie['genres'].split('|'))
                rec_genres = set(str(rec_row.get('genres', '')).split('|'))
                common = seed_genres.intersection(rec_genres)
                if common:
                    explanation_parts.append(f"Common genres: {', '.join(list(common)[:3])}.")
    
    # 4. Plot / Overview teaser (if available)
    overview = rec_row.get('overview', '')
    if isinstance(overview, str) and len(overview) > 30:
        short_overview = overview[:180].strip()
        if not short_overview.endswith('.'):
            short_overview += '...'
        explanation_parts.append(f"Plot hint: {short_overview}")
    
    # 5. Fun / Diversity note (random light touch)
    fun_phrases = [
        "This one stands out from the usual suggestions.",
        "A fresh pick that balances your past likes with new flavors.",
        "Users who enjoyed your favorites also loved this.",
        "Hidden gem alert!"
    ]
    explanation_parts.append(random.choice(fun_phrases))
    
    # Combine into one nice paragraph
    full_explanation = " ".join(explanation_parts)
    
    return full_explanation



In [6]:
# Integrating Explanation to the Hybrid Functions.
from sklearn.metrics.pairwise import cosine_similarity

def get_hybrid_recommendations_with_explanations(user_id, seed_movie_title=None, top_n=10, alpha=0.6):
    """Hybrid recommendations + explanations."""
    
    # Reuse hybrid logic from hybrid file - simplified here for completeness
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    all_movie_ids = enriched_movies['movieId'].unique()
    unrated_movies = [mid for mid in all_movie_ids if mid not in user_rated]
    
    collab_preds = {}
    for movie_id in unrated_movies[:4000]:   
        pred = svd_model.predict(uid=user_id, iid=movie_id)
        collab_preds[movie_id] = pred.est
    
    content_scores = {}
    seed_idx = None
    if seed_movie_title:
        seed_matches = enriched_movies[enriched_movies['title'].str.contains(seed_movie_title, case=False)]
        if len(seed_matches) > 0:
            seed_idx = seed_matches.index[0]
            seed_embedding = movie_embeddings[seed_idx]
            similarities = cosine_similarity([seed_embedding], movie_embeddings)[0]
            for i, sim in enumerate(similarities):
                mid = enriched_movies.iloc[i]['movieId']
                content_scores[mid] = float(sim)
    
    # Build hybrid results
    results = []
    for movie_id, collab_score in collab_preds.items():
        norm_collab = collab_score / 5.0
        content_score = content_scores.get(movie_id, 0.5)
        
        hybrid_score = (alpha * content_score) + ((1 - alpha) * norm_collab)
        
        row = enriched_movies[enriched_movies['movieId'] == movie_id].iloc[0]
        
        results.append({
            'movieId': movie_id,
            'title': row['title'],
            'hybrid_score': round(hybrid_score, 4),
            'content_score': round(content_score, 4),
            'collab_pred': round(collab_score, 2),
            'overview': row.get('overview', ''),
            'genres': row.get('genres', '')
        })
    
    # Sort and take top
    results.sort(key=lambda x: x['hybrid_score'], reverse=True)
    top_results = results[:top_n + 5]   # Extra for diversity later
    
    # Add explanations
    for rec in top_results:
        rec['explanation'] = generate_explanation(rec, user_id, seed_movie_title, alpha)
    
    return pd.DataFrame(top_results[:top_n])

## 3. Testing 

In [7]:
demo1 = get_hybrid_recommendations_with_explanations(user_id=1, seed_movie_title=None, top_n=5, alpha=0.5)
for i, row in demo1.iterrows():
    print(f"\n{i+1}. **{row['title']}**")
    print(f"   Hybrid Score: {row['hybrid_score']}")
    print(f"   Explanation: {row['explanation']}")

demo2 = get_hybrid_recommendations_with_explanations(user_id=1, seed_movie_title="Toy Story", top_n=5, alpha=0.7)
for i, row in demo2.iterrows():
    print(f"\n{i+1}. **{row['title']}**")
    print(f"   Hybrid Score: {row['hybrid_score']}")
    print(f"   Explanation: {row['explanation']}")


1. **Shawshank Redemption, The (1994)**
   Hybrid Score: 0.75
   Explanation: **Strong recommendation** with a hybrid score of 75%. Users with similar taste to you (user 1) rated similar movies very highly — we predict you'd give this **5.0/5**. Plot hint: Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting ski... Hidden gem alert!

2. **Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)**
   Hybrid Score: 0.75
   Explanation: **Strong recommendation** with a hybrid score of 75%. Users with similar taste to you (user 1) rated similar movies very highly — we predict you'd give this **5.0/5**. Plot hint: After the insane General Jack D. Ripper initiates a nuclear strike on the Soviet Union, a war room full of politicians, generals and a Russian diplomat all frantically try to stop... Users who enjoyed your favorites also loved this.

3. **

## 4. Saving the Explanation Utilities

In [8]:
explanation_dir = '../artifacts'
os.makedirs(explanation_dir, exist_ok=True)

explanation_config = {
    'version': '1.0',
    'description': 'Natural language explanations combining hybrid score, content themes, and collaborative signals'
}

config_path = os.path.join(explanation_dir, 'explanation_config.pkl')
with open(config_path, 'wb') as f:
    pickle.dump(explanation_config, f)
